# 06. Decision Trees and Random Forests — IMDb Spoiler Detection

This notebook contains a **tree-model analysis**. We use the same data preparation, grouped cross-validation, and evaluation conventions as [05_logistic_regression.ipynb](05_logistic_regression.ipynb).

**Workflow:**  
Tune with 10,000 training reviews and all eight input blocks  
→ backward-eliminate blocks using the first winning parameters  
→ retune on the complete *training* partition  
→ refit and save  
→ evaluate on the held-out *test* partition.  

During development, we compare a majority-class baseline, an untuned decision tree, a tuned decision tree, and a random forest.  
For the final held-out test, we evaluate using the majority baseline and the two tuned models.
The tuned tree and forest each select their **own** feature subset.  
No model receives `is_spoiler` or `movie_id` as an input feature.

## 1. Setup

Use the prepared train/test split from the project. The first two tuning stages use a reproducible 10,000-review sample, and the later stages use **all** training rows.  

We fix TF-IDF here, allowing us to focus on tree hyperparameters. Each text block gets its own vocabulary with up to 5,000 terms.

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.tree import plot_tree

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").is_dir(), "Open the notebook from the repository root or notebooks/."
sys.path.insert(0, str(PROJECT_ROOT))

from src.prepare_features import INITIAL_FEATURES, prepare_features
from src.split_data import RANDOM_STATE
from src.metrics import evaluate
from src.tree_models import (
    load_training_data, tune_tree_model, backward_elimination,
    refit_tree_model, get_feature_names,
)

sample_size = 10000
k = 5
group_by = "review_text"             # Standard review-grouped evaluation
initial_features = INITIAL_FEATURES.copy()
oversample_positive = True
max_macro_f1_drop = 0.005
forest_n_jobs = 2

cv_options = dict(
    k=k,
    group_by=group_by,
    random_state=RANDOM_STATE,
    oversample_positive=oversample_positive,
    n_jobs=forest_n_jobs,
)

tfidf_params = {"ngram_range": (1, 1), "min_df": 3, "max_features": 5000}
tree_grid = {"max_depth": [10, 30], "min_samples_leaf": [5, 20]}
forest_grid = {
    "n_estimators": [100], "max_features": ["sqrt"],
    "max_depth": [30, 60], "min_samples_leaf": [1, 5],
}

train_path = PROJECT_ROOT / "data/splits/IMDB_reviews_train.json"
test_path = PROJECT_ROOT / "data/splits/IMDB_reviews_test.json"
movies_path = PROJECT_ROOT / "data/cleaned/IMDB_movie_details.json"
output_dir = PROJECT_ROOT / "data/models/tree_models_review_text_simple"
output_dir.mkdir(parents=True, exist_ok=True)

for path in [train_path, test_path, movies_path]:
    assert path.is_file(), f"Missing prepared data: {path}"

## 2. Load the development sample

`load_training_data()`:
- chooses row positions using seed 67
- reads the reviews in chunks
- calls the shared `prepare_features()` to join movie information.

We retain labels and movie IDs for training and grouping.

In [ ]:
model_data, selection_rows, total_training_rows = load_training_data(
    train_path, movies_path, sample_size=sample_size, random_state=RANDOM_STATE,
)
print(f"Using {len(model_data):,} of {total_training_rows:,} training reviews")
print(f"Spoiler proportion: {model_data['is_spoiler'].mean():.2%}")
display(model_data[initial_features + ["is_spoiler"]].head(3))
np.save(output_dir / "selection_row_indices.npy", selection_rows)

## 3. Initial tuning: all eight blocks and five grouped folds

Each call to `tune_tree_model()` generates five folds.  

Within **each fold**, TF-IDF and other encoders are fitted on its training rows:
- positive training reviews oversampled only for the supervised models
- The original validation rows stay unchanged


- The **majority** model always predicts the training majority class.
- The **untuned tree** shows what sklearn's default tree achieves.
- The **tuned tree** compares four depth/leaf-size combinations.
    - We restrict maximum depth to 10 or 30 to avoid computationally expensive unrestricted-depth trees
- The **forest** compares four combinations, with 100 trees each.

We rank tuning candidates by **mean validation macro-F1**.  

`report_training=True` also measures training macro-F1 on the *original* (not duplicated) rows to help us inspect overfitting.

In [ ]:
majority_summary, _, _ = tune_tree_model(
    model_data, "majority", {}, tfidf_params,
    feature_columns=initial_features, report_training=True, verbose=False, **cv_options,
)
default_summary, _, _ = tune_tree_model(
    model_data, "decision_tree", {}, tfidf_params,
    feature_columns=initial_features, report_training=True, verbose=False, **cv_options,
)
tree_summary, tree_folds, tree_initial = tune_tree_model(
    model_data, "decision_tree", tree_grid, tfidf_params,
    feature_columns=initial_features, report_training=True, verbose=False, **cv_options,
)
forest_summary, forest_folds, forest_initial = tune_tree_model(
    model_data, "random_forest", forest_grid, tfidf_params,
    feature_columns=initial_features, report_training=True, verbose=False, **cv_options,
)
assert tree_initial is not None and forest_initial is not None
print("Initial winning tree settings:", tree_initial["model"])
print("Initial winning forest settings:", forest_initial["model"])
display(tree_summary)
display(forest_summary)

tree_summary.to_csv(output_dir / "initial_tree_cv.csv")
forest_summary.to_csv(output_dir / "initial_forest_cv.csv")
tree_folds.to_csv(output_dir / "initial_tree_folds.csv", index=False)
forest_folds.to_csv(output_dir / "initial_forest_folds.csv", index=False)

### Compare the four all-feature reference models

These scores come from the **same 10,000-review sample** with all eight input blocks.  

We compare validation macro-F1, spoiler precision/recall, fitting time, and the difference between training and validation macro-F1.  

A large positive gap can indicate overfitting, whilst a small gap is not useful if both scores are poor.

In [ ]:
reference_rows = [
    ("Majority", majority_summary.iloc[0]),
    ("Untuned tree", default_summary.iloc[0]),
    ("Tuned tree", tree_summary.loc[tree_summary["valid"]].iloc[0]),
    ("Random forest", forest_summary.loc[forest_summary["valid"]].iloc[0]),
]
comparison = pd.DataFrame([
    {"model": name,
     "train_macro_f1": row["train_macro_f1_mean"],
     "validation_macro_f1": row["macro_f1_mean"],
     "spoiler_precision": row["precision_mean"],
     "spoiler_recall": row["recall_mean"],
     "fit_seconds_per_fold": row["fit_seconds_mean"]}
    for name, row in reference_rows
]).set_index("model")
comparison["train_validation_gap"] = (
    comparison["train_macro_f1"] - comparison["validation_macro_f1"]
)
display(comparison.round(4))
comparison.to_csv(output_dir / "initial_comparison.csv")
comparison[["train_macro_f1", "validation_macro_f1"]].plot(
    kind="bar", figsize=(9, 4), ylim=(0, 1), title="Initial all-feature comparison"
)
plt.ylabel("Macro-F1")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

- The untuned tree shows substantial overfitting
    - Training macro-F1 is 1.0000, but validation macro-F1 is only 0.5587
    - Restricting the tree to depth 10 and at least 20 samples per leaf reduced the training–validation gap from 0.4413 to 0.1191, while improving validation macro-F1 to 0.5736
        - This suggests regularization may be a useful approach for this model

- The forest has a different precision–recall trade-off
    - Initially tuned forest and tree have almost identical macro-F1 scores, but very different spoiler-class metrics
        - Tuned tree: precision 35.57%, recall 52.47%
        - Random forest: precision 48.69%, recall 22.82%
    - Forest is more reliable when it flags a review, but it misses substantially more actual spoilers at the fixed classification threshold

## 4. Backward feature elimination

- Hold **each model family's initial winning hyperparameters fixed**
- Starting with all eight blocks, evaluate the removal of each block repeatedly
    - Follow the removal with the best validation macro-F1
- Finally, choose the **smallest visited subset** within an absolute `0.005` of the best validation macro-F1 on that path.

Note: This removes *whole blocks* such as `review_summary` (not individual TF-IDF terms).  

The dataframe still retains `review_text` for grouping even if that block is removed from model inputs.  

The tree and forest may choose different subsets.

In [ ]:
tree_features, tree_path, tree_trials = backward_elimination(
    model_data, "decision_tree", tree_initial,
    max_macro_f1_drop=max_macro_f1_drop, **cv_options,
)
forest_features, forest_path, forest_trials = backward_elimination(
    model_data, "random_forest", forest_initial,
    max_macro_f1_drop=max_macro_f1_drop, **cv_options,
)
print("Tree selected:", tree_features)
print("Forest selected:", forest_features)
display(tree_path)
display(forest_path)

tree_path.to_csv(output_dir / "tree_elimination_path.csv", index=False)
forest_path.to_csv(output_dir / "forest_elimination_path.csv", index=False)
tree_trials.to_csv(output_dir / "tree_elimination_trials.csv", index=False)
forest_trials.to_csv(output_dir / "forest_elimination_trials.csv", index=False)

# Save just the choices needed by the later stages.
development = {"decision_tree": tree_features, "random_forest": forest_features}
joblib.dump(development, output_dir / "development_features.joblib")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for label, path in [("Decision tree", tree_path), ("Random forest", forest_path)]:
    ax.plot(path["n_blocks"], path["macro_f1_mean"], marker="o", label=label)
ax.invert_xaxis() # Show the elimination path from eight blocks down to one
ax.set(xlabel="Number of feature blocks kept", ylabel="Mean validation macro-F1")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Retune on the full training partition

This is a separate, larger search.  

Use the **selected blocks** from the development sample, but rerun each family's original hyperparameter grid on five review-grouped folds of *all training reviews*.  

We don't reuse the initial winning hyperparameters automatically.  

This reads only `IMDB_reviews_train.json`, never the held-out test split.

In [ ]:
# To resume here after a kernel restart, rerun section 1, then this cell.
development = joblib.load(output_dir / "development_features.joblib")
full_model_data, _, full_row_count = load_training_data(
    train_path, movies_path, sample_size=None, random_state=RANDOM_STATE,
)
assert len(full_model_data) == full_row_count
print(f"Full-training CV: {len(full_model_data):,} original reviews")

In [ ]:
selection_file = output_dir / "full_selection.joblib"

# Load previously completed model searches, if any.
full_best = joblib.load(selection_file) if selection_file.exists() else {}

for kind, grid in [
    ("decision_tree", tree_grid),
    ("random_forest", forest_grid),
]:
    if kind in full_best:
        print(f"Already completed: {kind}")
        continue

    selected = development[kind]

    summary, folds, best = tune_tree_model(
        full_model_data,
        kind,
        grid,
        tfidf_params,
        feature_columns=selected,
        verbose=True,
        **cv_options,
    )

    if best is None:
        raise RuntimeError(f"No usable full-training CV candidate for {kind}")

    full_best[kind] = best

    summary.to_csv(output_dir / f"full_{kind}_cv.csv")
    folds.to_csv(output_dir / f"full_{kind}_folds.csv", index=False)

    # Save immediately after this model family finishes.
    joblib.dump(full_best, selection_file)

    print(f"{kind}: selected blocks={selected}, final params={best['model']}")
    display(summary.head())

## 6. Fit and save the final pipelines

Cross-validation only **selects** hyperparameters and feature blocks. We now fit fresh feature encoders and classifiers using **every original training review**.  

We save each fitted encoder and model together as a sklearn `Pipeline`.  

For final evaluation, we refit the majority baseline alongside the selected decision tree and random forest. The untuned tree is retained as a development-stage reference but is excluded from full-data fitting due to its computational cost.  

The dummy baseline uses the natural (not oversampled) class proportions.

In [ ]:
# If resuming after a restart, run section 1 and the full-data loading cell first.
full_best = joblib.load(output_dir / "full_selection.joblib")
models_to_fit = {
    "majority": ("majority", {"model": {}, "features": initial_features, "tfidf": tfidf_params}),
    # "untuned_tree": ("decision_tree", {"model": {}, "features": initial_features, "tfidf": tfidf_params}), # This is computationally expensive and unnecessary (we already observed substantial overfitting from untuned tree)
    "decision_tree": ("decision_tree", full_best["decision_tree"]),
    "random_forest": ("random_forest", full_best["random_forest"]),
}

for name, (kind, best) in models_to_fit.items():

    save_path = output_dir / f"{name}.joblib"

    # Avoid refitting models that have already been saved.
    if save_path.exists():
        print(f"Already saved {name}, skipping")
        continue

    print(f"Fitting {name}...", flush=True)

    model = refit_tree_model(
        full_model_data,
        kind,
        best,
        random_state=RANDOM_STATE,
        oversample_positive=oversample_positive,
        n_jobs=forest_n_jobs,
    )

    print(f"Saving {name}...", flush=True)

    joblib.dump({
        "pipeline": model,
        "features": best["features"],
        "params": best["model"],
        "tfidf": best["tfidf"],
        "group_by": group_by
        },
        save_path, compress=3
        )
    print(f"Saved {name} ({len(best['features'])} feature blocks)")
    del model

## 7. Inspect the selected decision tree

We use tree plot to display the first few levels of the fitted model without altering its depth.  

We use a feature-importance table to measure the model's impurity reductions.  

Terms are prefixed with their source field so that e.g. `review_text: killer` and `plot_summary: killer` are distinct.

In [ ]:
saved_tree = joblib.load(output_dir / "decision_tree.joblib")["pipeline"]
feature_names = get_feature_names(saved_tree.named_steps["features"])
tree = saved_tree.named_steps["classifier"]

plt.figure(figsize=(18, 7))
plot_tree(tree, feature_names=list(feature_names), class_names=["Non-spoiler", "Spoiler"],
          max_depth=2, filled=True, fontsize=8)
plt.title("First three levels of the fitted decision tree")
plt.tight_layout()
plt.show()

importance = pd.DataFrame({
    "feature": feature_names, "importance": tree.feature_importances_,
}).sort_values("importance", ascending=False)
display(importance.head(20))
importance.to_csv(output_dir / "decision_tree_feature_importance.csv", index=False)

## 8. Final evaluation on the **held-out test set**

This is only run after the feature subsets and hyperparameters are finalized.  

The saved pipelines apply their own fitted encoders to each test chunk.  

We calculate metrics over **all test reviews** (not by averaging the F1 scores of chunks).  

Note: the `> 0.5` spoiler-score rule matches the project's previous evaluation convention.

In [ ]:
movies = pd.read_json(movies_path, lines=True, convert_dates=False)
results = []

for name in models_to_fit:
    saved = joblib.load(output_dir / f"{name}.joblib")
    predictions = []
    with pd.read_json(test_path, lines=True, convert_dates=False, chunksize=10000) as reader:
        for reviews in reader:
            X_test = prepare_features(reviews, movies)[saved["features"]]
            probability = saved["pipeline"].predict_proba(X_test)[:, 1]
            predictions.append(pd.DataFrame({
                "test_row": reviews.index.to_numpy(),
                "movie_id": reviews["movie_id"].to_numpy(),
                "is_spoiler": reviews["is_spoiler"].to_numpy().astype(int),
                "predicted_spoiler": (probability > 0.5).astype(int),
                "spoiler_probability": probability,
            }))

    predictions = pd.concat(predictions, ignore_index=True)
    scores = evaluate(predictions["is_spoiler"], predictions["predicted_spoiler"],
                      predictions["spoiler_probability"])
    scores["average_precision"] = scores.pop("pr_auc")
    results.append({"model": name, **scores})
    confusion = pd.DataFrame(
        [[scores["tn"], scores["fp"]], [scores["fn"], scores["tp"]]],
        index=["Actual non-spoiler", "Actual spoiler"],
        columns=["Predicted non-spoiler", "Predicted spoiler"],
    )
    confusion.to_csv(output_dir / f"{name}_test_confusion_matrix.csv")
    predictions.to_csv(output_dir / f"{name}_test_predictions.csv", index=False)
    print(f"{name}: test macro-F1={scores['macro_f1']:.4f}, "
          f"spoiler recall={scores['recall']:.4f}")
    del saved

comparison = pd.DataFrame(results).set_index("model")
display(comparison[["macro_f1", "precision", "recall", "f1", "average_precision", "accuracy"]])
comparison.to_csv(output_dir / "test_comparison.csv")

## 9. Observations and Discussion

### 9.1 Model Complexity and Overfitting

The initial experiments used 10,000 training reviews, of which 26.03% were labelled as spoilers. The majority-class baseline achieved a mean validation macro-F1 of 0.4252 but had zero spoiler recall, demonstrating why accuracy alone is insufficient for this imbalanced classification task.

The untuned decision tree exhibited substantial overfitting, achieving a training macro-F1 of 1.0000 but a mean validation macro-F1 of only 0.5587. Restricting the tree to `max_depth=10` and `min_samples_leaf=20` improved validation macro-F1 to 0.5736 while reducing training macro-F1 to 0.6927. The training–validation gap consequently decreased from 0.4413 to 0.1191, suggesting that restricting tree complexity improved generalization.

The initially tuned random forest achieved a similar validation macro-F1 of 0.5731, despite a much higher training macro-F1 of 0.9820. This suggests that the forest also fitted the development training folds much more closely than the validation folds.

### 9.2 Backward feature elimination

Backward elimination was performed separately for both models using five-fold grouped cross-validation and fixed initial hyperparameters.

For the decision tree, the highest mean validation macro-F1 along the elimination path was 0.5863 with five feature blocks. However, retaining only `review_text` achieved 0.5814, a difference of approximately 0.0049. Because this was within the predefined 0.005 tolerance, the simpler one-block subset was selected.

For the random forest, the highest mean validation macro-F1 was 0.6088 using `review_text` and `duration`. Removing `duration` reduced macro-F1 to 0.6018, a decrease of approximately 0.0070, exceeding the allowed tolerance. Thus, the two-block subset was retained.

Both models benefited from removing several original feature blocks during development. Review text remained in both selected subsets, suggesting that it contained useful predictive information. However, greedy backward elimination does not evaluate every possible feature combination and cannot guarantee a globally optimal subset.

### 9.3 Full-training hyperparameter retuning

The selected feature subsets were subsequently used for five-fold grouped cross-validation on all 459,130 original training reviews.

The decision tree selected `max_depth=10` and `min_samples_leaf=5`, achieving a mean validation macro-F1 of 0.6175 (standard deviation 0.0022).

The random forest selected `max_depth=30`, `min_samples_leaf=5`, `max_features="sqrt"`, and `n_estimators=100`, achieving a higher mean validation macro-F1 of 0.6535 (standard deviation 0.0023).

The forest was substantially more computationally expensive. For the winning configurations, mean classifier-fitting time was approximately 625.0 seconds per CV fold for the forest, compared with 75.5 seconds for the tree. These measurements exclude feature-preprocessing time.

Unrestricted-depth trees were excluded from the revised tuning grid due to computational cost. The untuned tree was also omitted from final full-data fitting, although its development results were retained for comparison. Consequently, the final test evaluation includes the majority baseline and the two tuned models, but not the untuned tree.

### 9.4 Held-out test performance

The final models were evaluated on the held-out test partition after feature selection and hyperparameter tuning had been completed.

| Metric | Majority | Decision tree | Random forest |
|---|---:|---:|---:|
| Accuracy | 0.7370 | 0.6554 | 0.7325 |
| Macro-F1 | 0.4243 | 0.6144 | **0.6563** |
| Spoiler precision | 0.0000 | 0.4006 | **0.4916** |
| Spoiler recall | 0.0000 | **0.6258** | 0.4973 |
| Spoiler F1 | 0.0000 | 0.4885 | **0.4944** |
| Average precision | 0.2630 | 0.4809 | **0.5171** |

The random forest achieved the highest test macro-F1 of 0.6563, outperforming the decision tree by approximately 0.0419. It also achieved higher spoiler precision, spoiler F1, and average precision.

However, the decision tree achieved substantially higher spoiler recall (0.6258 versus 0.4973). This indicates a trade-off: at the fixed 0.5 classification threshold, the tree identified a larger proportion of actual spoilers but produced less precise spoiler predictions.

The majority baseline achieved the highest accuracy of 0.7370, slightly exceeding the forest's 0.7325, despite predicting no spoilers correctly. This illustrates why accuracy can be misleading for an imbalanced dataset. Macro-F1 provides a more informative comparison because it considers performance on both classes.

The final test macro-F1 scores were also close to their full-training CV means: 0.6144 versus 0.6175 for the tree, and 0.6563 versus 0.6535 for the forest. This suggests that the selected configurations performed consistently between cross-validation and the held-out test under the project's evaluation split.

Overall, the forest provided the strongest balanced classification performance, while the decision tree was preferable specifically for spoiler recall at the evaluated threshold.

### 9.5 Decision-tree interpretation

The final decision tree uses only TF-IDF features derived from `review_text`.

Its highest impurity-based feature importances included:

- `review_text: he` — 0.2420
- `review_text: spoilers` — 0.1462
- `review_text: spoiler` — 0.1171
- `review_text: they` — 0.0741
- `review_text: when` — 0.0532
- `review_text: her` — 0.0510
- `review_text: end` — 0.0426

The visualized tree splits first on the TF-IDF value of `he`, with `spoilers` appearing in subsequent splits. Explicit terms such as `spoiler`, `spoilers`, and `end` appear alongside common pronouns and other general words.

These findings indicate that the model uses lexical patterns associated with the training labels. However, impurity-based feature importance does not establish causation, and the presence of these terms does not demonstrate that the model understands narrative events or plot revelations. In particular, reliance on explicit spoiler-related words may differ from the ability to detect subtle spoilers without such cues.

Only the selected decision tree's feature importances were examined in this notebook; equivalent feature-importance conclusions cannot be drawn about the forest from the displayed results.

### 9.6 Limitations

The experiments used unigram TF-IDF features, which do not directly represent broader semantic context or narrative structure.

Cross-validation grouped reviews by `review_text` to prevent identical review texts from appearing across training and validation folds. However, different reviews of the same movie may still appear in different partitions. The evaluation therefore does not establish performance on entirely unseen movies.

The tree and forest also used different selected feature subsets, so their performance difference cannot be attributed exclusively to the classification algorithms.

Finally, all reported test predictions used the project's fixed spoiler-probability threshold of 0.5. Changing this threshold could alter the precision–recall trade-off, but no threshold optimization was conducted in these experiments. The held-out test results were used for final evaluation rather than further model selection.